# Train Logistic Regression Core

This notebook keeps only the essential steps needed to train a logistic regression model with scikit-learn.
It is meant to show how much of the machine learning workflow is already wrapped in built-in library classes.


In [ ]:
# =========================
# Notebook setup and imports
# =========================
!pip install pandas
!pip install -U scikit-learn

from __future__ import annotations

import pickle
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


In [ ]:
# ======================
# Minimal config
# ======================

data_path = Path("WA_Fn-UseC_-Telco-Customer-Churn.csv")
model_out = Path("logistic_regression_churn_core.pkl")
test_size = 0.2
random_state = 42

In [ ]:
# ======================
# Training Data Loading
# ======================

def load_data(data_path: Path) -> tuple[pd.DataFrame, pd.Series]:
    data = pd.read_csv(data_path)

    # Keep only useful modeling columns and convert the target to 0/1.
    data = data.drop(columns=["customerID"])
    data["TotalCharges"] = pd.to_numeric(data["TotalCharges"], errors="coerce")
    data["Churn"] = data["Churn"].map({"No": 0, "Yes": 1})

    X = data.drop(columns=["Churn"])
    y = data["Churn"]
    return X, y

In [ ]:
# ==================
# Model Build
# ==================
def build_model(
    numeric_features: list[str],
    categorical_features: list[str],
) -> Pipeline:
    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_pipeline, numeric_features),
            ("cat", categorical_pipeline, categorical_features),
        ]
    )

    return Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("classifier", LogisticRegression(max_iter=1000)),
        ]
    )

In [ ]:

# Main Processes

X, y = load_data(data_path)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=test_size,
    random_state=random_state,
    stratify=y,
)

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()
model = build_model(numeric_features, categorical_features)

model.fit(X_train, y_train)

predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

print(f"Training rows: {len(X_train)}")
print(f"Test rows: {len(X_test)}")
print(f"Numeric features: {numeric_features}")
print(f"Categorical feature count: {len(categorical_features)}")
print(f"Accuracy: {accuracy:.4f}")
print("\nClassification report:")
print(classification_report(y_test, predictions, target_names=["No Churn", "Churn"]))

with model_out.open("wb") as model_file:
    pickle.dump(model, model_file)

print(f"Saved trained pipeline to: {model_out}")
